In [1]:
import cv2
import numpy as np
import tensorflow as tf
import time

model = tf.keras.models.load_model("emotion_model.keras")

In [2]:
import random

class_names = [
    "angry",
    "fearful",
    "happy",
    "neutral",
    "sad",
    "surprised"
]

transition_responses = {

    ("neutral", "happy"): [
        "That smile wasn't there a moment ago. Something good happened?",
        "Looks like your mood just improved!",
        "You seem much happier now."
    ],

    ("sad", "happy"): [
        "I'm glad to see you smiling again.",
        "Looks like things are getting better.",
        "Your smile is back!"
    ],

    ("happy", "sad"): [
        "You don't seem as cheerful anymore.",
        "I hope everything is okay.",
        "Did something happen?"
    ],

    ("neutral", "sad"): [
        "You seem a little down today.",
        "Hope your day gets better.",
        "Take things one step at a time."
    ],

    ("neutral", "fearful"): [
        "You seem a little concerned.",
        "Everything alright?",
    ],

    ("fearful", "neutral"): [
        "Looks like you've relaxed a little.",
        "Glad to see you're feeling calmer."
    ],

    ("neutral", "surprised"): [
        "Something caught your attention!",
        "That was unexpected, wasn't it?"
    ]
}

stable_responses = {

    "happy": [
        "You're still smiling! Keep it up.",
        "It's nice seeing you in a good mood.",
        "Hope your day keeps getting better."
    ],

    "neutral": [
        "What's on your mind?",
        "Hope you're having a good day.",
        "Anything interesting happening today?"
    ],

    "sad": [
        "Remember to take breaks when you need them.",
        "Tomorrow is another day.",
        "Hope things get better soon."
    ],

    "angry": [
        "Maybe taking a deep breath would help.",
        "Hope things calm down soon."
    ],

    "fearful": [
        "Everything will be alright.",
        "Take things one step at a time."
    ],

    "surprised": [
        "That certainly caught your attention!",
        "Didn't expect that?"
    ]
}

In [3]:
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [4]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise Exception("Cannot open webcam")

In [5]:
detection_interval = 0.2
last_detection = 0

last_face = None

display_emotion = "Detecting..."
display_confidence = 0.0

assistant_message = "Hello!"

# -------- EMA smoothing --------

smoothed_prediction = None
alpha = 0.4

# -------- Emotion stability --------

current_emotion = None
previous_emotion = None

emotion_start_time = time.time()

stable_duration = 3
last_message_time = 0
message_cooldown = 8

In [6]:
while True:

    ret, frame = cap.read()

    if not ret:
        break

    current_time = time.time()

    # -------------------------------------------------
    # Detect every 0.2 second
    # -------------------------------------------------

    if current_time - last_detection >= detection_interval:

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(80,80)
        )

        if len(faces) > 0:

            last_face = max(
                faces,
                key=lambda b: b[2]*b[3]
            )

            x, y, w, h = last_face

            face = frame[y:y+h, x:x+w]

            face = cv2.resize(face, (48,48))

            face = cv2.cvtColor(
                face,
                cv2.COLOR_BGR2RGB
            )

            face = face.astype(np.float32)

            input_tensor = np.expand_dims(face, axis=0)

            prediction = model.predict(
                input_tensor,
                verbose=0
            )[0]

            # -----------------------------------------
            # EMA smoothing
            # -----------------------------------------

            if smoothed_prediction is None:
                smoothed_prediction = prediction
            else:
                smoothed_prediction = (
                    alpha * prediction +
                    (1-alpha) * smoothed_prediction
                )

            emotion_idx = np.argmax(smoothed_prediction)

            confidence = float(
                smoothed_prediction[emotion_idx]
            )

            if confidence >= 0.60:

                detected_emotion = class_names[emotion_idx]

                display_emotion = detected_emotion
                display_confidence = confidence

                # -------------------------------------
                # Emotion changed
                # -------------------------------------
                
                if detected_emotion != current_emotion:
                
                    previous_emotion = current_emotion
                    current_emotion = detected_emotion
                
                    emotion_start_time = current_time
                
                    transition = (previous_emotion, current_emotion)
                
                    if transition in transition_responses:
                        assistant_message = random.choice(
                            transition_responses[transition]
                        )
                    else:
                        assistant_message = random.choice(
                            stable_responses[current_emotion]
                        )
                
                    last_message_time = current_time
                
                # -------------------------------------
                # Emotion stayed stable
                # -------------------------------------
                
                elif (
                    current_time - emotion_start_time >= stable_duration
                    and
                    current_time - last_message_time >= message_cooldown
                ):
                
                    assistant_message = random.choice(
                        stable_responses[current_emotion]
                    )
                
                    last_message_time = current_time
        last_detection = current_time

    # -------------------------------------------------
    # Draw
    # -------------------------------------------------

    if last_face is not None:

        x, y, w, h = last_face

        cv2.rectangle(
            frame,
            (x,y),
            (x+w,y+h),
            (0,255,0),
            2
        )

        label = f"{display_emotion} ({display_confidence*100:.1f}%)"
        position = (x, y - 15)
        
        # Outline
        cv2.putText(
            frame,
            label,
            position,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 0, 0),
            5,
            cv2.LINE_AA
        )
        
        # Green text
        cv2.putText(
            frame,
            label,
            position,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )
        # Assistant message

        text = assistant_message
        position = (20, 40)
        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.7
        thickness = 2
        
        # Black outline
        cv2.putText(
            frame,
            text,
            position,
            font,
            scale,
            (0, 0, 0),
            thickness + 3,
            cv2.LINE_AA
        )
        
        # White text
        cv2.putText(
            frame,
            text,
            position,
            font,
            scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA
        )

    cv2.imshow(
        "Emotion Assistant",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Untuk progress kali ini, diimplementasikan model tersebut untuk real time detection menggunakan webcam. Disini, juga dicoba menggunakan google Gemini untuk membuatnya lebih interaktif lagi. Untuk saat ini masih belum berhasil disambungkan dengan Gemini jadi hasilnya hanya menunjukkan teks default berdasarkan ekspresi saja. Rencana berikutnya adalah untuk menyelesaikannya dan mendeploy proyek tersebut.